In [20]:
import pandas as pd
import re
from pathlib import Path

def clean_text_strict(text):
    """
    彻底清洗文本：只保留纯英文、数字和核心标点。
    """
    if not isinstance(text, str):
        return ""
    
    # 1. 去除 URL
    text = re.sub(r'http[s]?://\S+', '', text)
    
    # 2. 强制转为 ASCII 并过滤 (白名单模式)
    # 只保留: a-z, A-Z, 0-9, 空格及基础标点 (.,!?;:'"() -)
    text = re.sub(r'[^a-zA-Z0-9\s.,!?;:\'"\-\(\)]', ' ', text)
    
    # 3. 合并多余空格和换行
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

def preprocess_daigt(csv_path):
    """
    预处理 DAIGT 数据集：标准化列名、严格清洗、1:1 均衡采样
    """
    df = pd.read_csv(csv_path)
    
    # 1. 列名标准化处理 (容错处理)
    df.columns = [col.strip().lower() for col in df.columns]
    
    # 建立映射：将原始列映射到我们统一的标准
    # DAIGT 常见列名为 generated (label), text (text), prompt_name (domain)
    column_mapping = {
        'generated': 'label',
        'prompt_name': 'prompt',
        'rdizzl3_seven': 'is_train'
    }
    df = df.rename(columns=column_mapping)
    
    print(f"原始数据: {len(df)} 行, 列: {df.columns.tolist()}")

    # 2. 严格清洗与长度过滤
    df['text'] = df['text'].apply(clean_text_strict)
    
    # 过滤掉清洗后的超短/超长样本 (100-5000字符)
    min_len, max_len = 100, 5000
    df = df[df['text'].str.len().between(min_len, max_len)].copy()
    
    # 3. 标签规范化
    if 'label' not in df.columns and 'generated' not in df.columns:
        # 如果没有 label 列，尝试寻找可能的标识列
        print("警告: 未找到 label 列，请检查 CSV 结构")
        return pd.DataFrame()
    
    df['label'] = df['label'].astype(int)

    # 4. ⚠️ 核心：AI / Human 严格 1:1 均衡采样
    label_counts = df['label'].value_counts()
    print(f"清洗过滤后的分布: {label_counts.to_dict()}")
    
    if len(label_counts) == 2:
        min_size = label_counts.min()
        df = df.groupby('label').apply(
            lambda x: x.sample(n=min_size, random_state=42)
        ).reset_index(drop=True)
        # 随机打乱全局顺序
        df = df.sample(frac=1, random_state=42).reset_index(drop=True)
        print(f">>> 均衡化完成：AI 和 Human 各保留 {min_size} 条样本")
    else:
        print(f">>> 警告：类别不足，当前分布: {label_counts.to_dict()}")

    # 5. 构造统一字段 (source/model/domain)
    dataset_name = "DAIGT"
    file_stem = Path(csv_path).stem.lower()
    split = 'train' if 'train' in file_stem else 'test'
    
    # 这里的 source 信息尽量保留原始多样性
    df['source'] = f"{dataset_name}_{split}"
    
    # DAIGT 中的 AI 主要是 GPT 系列生成，Human 则是学生作文
    df['model'] = df['label'].apply(lambda x: 'gpt-3.5-turbo' if x == 1 else 'human')
    
    # 领域处理：从 prompt 名称中提取
    if 'prompt' in df.columns:
        df['domain'] = df['prompt'].fillna('essay').apply(
            lambda x: re.sub(r'\W+', '_', str(x).lower())[:30]
        )
    else:
        df['domain'] = 'essay'

    # 6. 最终输出标准格式
    final_df = df[['text', 'label', 'source', 'model', 'domain']].copy()

    print(f"\nDAIGT 预处理成功")
    print(f"最终标签分布: {final_df['label'].value_counts().to_dict()}")
    
    return final_df

# ========================================
# 主程序
# ========================================
if __name__ == "__main__":
    CSV_PATH = r"D:\Desktop\capstone\code\datasets\DAIGT.csv"
    SAVE_DIR = Path(r"D:\Desktop\capstone\code\datasets\processed")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    df_daigt = preprocess_daigt(CSV_PATH)

    if not df_daigt.empty:
        output_path = SAVE_DIR / "DAIGT_Clean_Balanced.csv"
        df_daigt.to_csv(output_path, index=False, encoding='utf-8')
        
        print("-" * 30)
        print(f"已保存: {output_path}")
        print(f"最终样本总数: {len(df_daigt)}")
    else:
        print("未生成任何有效数据。")

原始数据: 44868 行, 列: ['text', 'label', 'prompt', 'source', 'is_train']
清洗过滤后的分布: {0: 26557, 1: 17494}
>>> 均衡化完成：AI 和 Human 各保留 17494 条样本

DAIGT 预处理成功
最终标签分布: {0: 17494, 1: 17494}


C:\Users\21063\AppData\Local\Temp\ipykernel_32732\1275926410.py:65: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('label').apply(


------------------------------
已保存: D:\Desktop\capstone\code\datasets\processed\DAIGT_Clean_Balanced.csv
最终样本总数: 34988


In [15]:
import pandas as pd
import re
import numpy as np
from pathlib import Path

def clean_text_strict(text):
    """
    彻底清洗文本：只保留纯英文、数字和基础标点。
    """
    if not isinstance(text, str):
        return ""
    
    # 1. 去除 URL
    text = re.sub(r'http[s]?://\S+', '', text)
    
    # 2. 严格去除非英语符号 (白名单模式)
    # 只保留: 字母, 数字, 空格, 以及基础标点 (.,!?;:'"() -)
    text = re.sub(r'[^a-zA-Z0-9\s.,!?;:\'"\-\(\)]', ' ', text)
    
    # 3. 合并多余空格和换行
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

def preprocess_multimodel_nuclear(file_path):
    """
    保留原始解析逻辑，并加入严格清洗和均衡处理
    """
    path = Path(file_path)
    # 读取原始文本内容
    text_content = path.read_text(encoding='utf-8')

    # --- 原始解析部分 ---
    lines = [line.strip() for line in text_content.split('\n') if line.strip()]
    if not lines:
        return pd.DataFrame()

    # 提取模型列表（假设在第二行）
    model_line = lines[1]
    models = [m.strip() for m in model_line.split() if m.strip()]

    sections = re.split(r'\n\s*\n', text_content)
    raw_data = []
    human_start = False

    for section in sections:
        section = section.strip()
        if not section: continue

        # 检测 Human 文章起点
        if any(marker in section.lower() for marker in ['by ', 'published:', 'the human toll']):
            human_start = True

        if human_start:
            # 排除 AI 的标题干扰
            if not any(ai_title in section for ai_title in ["The Unseen Scars", "The Atomic Aftermath", "The Ghostly Whispers"]):
                raw_data.append({
                    'text': section,
                    'label': 0,
                    'model': 'human'
                })
                human_start = False # 假设一段就是一个 human 样本
        else:
            # 匹配 AI 模型输出
            for model in models:
                if model.lower() in section.lower() and len(section) > 100:
                    model_text = section.split(model, 1)[1] if model in section else section
                    # 截断到下一个可能的标题
                    next_title = re.search(r'(Title:|# |\*\*By|## |<end_of_turn>)', model_text)
                    if next_title:
                        model_text = model_text[:next_title.start()]
                    
                    raw_data.append({
                        'text': model_text,
                        'label': 1,
                        'model': model
                    })
                    break

    df = pd.DataFrame(raw_data)
    
    # --- 彻底预处理与过滤 ---
    # 1. 严格清洗
    df['text'] = df['text'].apply(clean_text_strict)
    
    # 2. 长度过滤 (去掉太短 < 100字符 和 太长 > 5000字符)
    min_len, max_len = 100, 5000
    df = df[df['text'].str.len().between(min_len, max_len)].copy()

    # 3. 类别均衡 (1:1 比例)
    label_counts = df['label'].value_counts()
    if len(label_counts) == 2:
        min_size = label_counts.min()
        df = df.groupby('label').apply(lambda x: x.sample(n=min_size, random_state=42)).reset_index(drop=True)
        print(f"均衡化完成，每类保留 {min_size} 条样本。")
    else:
        print(f"警告：类别不全，当前分布: {label_counts.to_dict()}")

    return df

if __name__ == "__main__":
    FILE_PATH = r"D:\Desktop\capstone\code\datasets\gsingh1-py_train.csv"
    SAVE_DIR = Path(r"D:\Desktop\capstone\code\datasets\processed")
    SAVE_DIR.mkdir(exist_ok=True, parents=True)

    # 调用处理函数
    df_final = preprocess_multimodel_nuclear(FILE_PATH)
    
    if not df_final.empty:
        output_path = SAVE_DIR / "MultiModel_Clean_Balanced.csv"
        df_final.to_csv(output_path, index=False, encoding='utf-8')
        
        print("\n" + "="*30)
        print(f"处理成功！")
        print(f"总样本数: {len(df_final)}")
        print(f"标签分布: {df_final['label'].value_counts().to_dict()}")
        print(f"文件保存至: {output_path}")
        print("="*30)
    else:
        print("未提取到任何有效数据，请检查文件内容格式。")

C:\Users\21063\AppData\Local\Temp\ipykernel_32732\3254711490.py:94: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('label').apply(lambda x: x.sample(n=min_size, random_state=42)).reset_index(drop=True)


均衡化完成，每类保留 71392 条样本。

处理成功！
总样本数: 142784
标签分布: {0: 71392, 1: 71392}
文件保存至: D:\Desktop\capstone\code\datasets\processed\MultiModel_Clean_Balanced.csv


In [17]:
import pandas as pd
import ast
import re
from pathlib import Path

def clean_text_strict(text):
    """
    彻底清洗文本：只保留纯英文、数字和基础标点。
    """
    if not isinstance(text, str):
        return ""
    
    # 1. 去除 URL (HC3 数据中常包含引用链接)
    text = re.sub(r'http[s]?://\S+', '', text)
    
    # 2. 严格去除非英语符号 (白名单模式)
    # 只保留: a-z, A-Z, 0-9, 空格, 以及基础标点 (.,!?;:'"() -)
    text = re.sub(r'[^a-zA-Z0-9\s.,!?;:\'"\-\(\)]', ' ', text)
    
    # 3. 合并多余空格和换行
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

def safe_eval_list(s):
    """安全解析字符串列表"""
    if pd.isna(s) or not s:
        return []
    try:
        return ast.literal_eval(s)
    except:
        s = s.strip("[]")
        return [item.strip("'\" ") for item in s.split(",") if item.strip("'\" ")]

def preprocess_hc3(csv_path):
    """
    处理 HC3 数据集：展开列表、严格清洗、长度过滤、类别均衡
    """
    df = pd.read_csv(csv_path)
    print(f"原始数据: {len(df)} 行")
    
    rows = []
    min_len, max_len = 100, 5000  # 设定严格长度区间

    for _, row in df.iterrows():
        source = str(row['source']) if pd.notna(row['source']) else "unknown"
        subset = str(row['subset']) if pd.notna(row['subset']) else "unknown"
        
        # 解析并处理人类答案
        human_ans = safe_eval_list(row['human_answers'])
        for ans in human_ans:
            cleaned = clean_text_strict(ans)
            if min_len <= len(cleaned) <= max_len:
                rows.append({
                    'text': cleaned,
                    'label': 0,
                    'source': f"HC3_{source}_{subset}",
                    'model': 'human'
                })
        
        # 解析并处理 ChatGPT 答案
        gpt_ans = safe_eval_list(row['chatgpt_answers'])
        for ans in gpt_ans:
            cleaned = clean_text_strict(ans)
            if min_len <= len(cleaned) <= max_len:
                rows.append({
                    'text': cleaned,
                    'label': 1,
                    'source': f"HC3_{source}_{subset}",
                    'model': 'gpt-3.5-turbo'
                })
    
    final_df = pd.DataFrame(rows)
    
    # --- 自动均衡处理 (1:1 比例) ---
    print(f"清洗与过滤后样本数: {len(final_df)}")
    label_counts = final_df['label'].value_counts()
    
    if len(label_counts) == 2:
        min_size = label_counts.min()
        # 随机下采样使两类数量一致
        final_df = final_df.groupby('label').apply(
            lambda x: x.sample(n=min_size, random_state=42)
        ).reset_index(drop=True)
        # 再次打乱顺序
        final_df = final_df.sample(frac=1, random_state=42).reset_index(drop=True)
        print(f">>> 类别均衡完成：Human={min_size}, AI={min_size}")
    else:
        print(f">>> 警告：类别不平衡，当前分布: {label_counts.to_dict()}")

    return final_df

if __name__ == "__main__":
    # 配置路径 (根据你的环境修改)
    CSV_PATH = r"D:\Desktop\capstone\code\datasets\HC3_all_merged.csv"
    SAVE_DIR = Path(r"D:\Desktop\capstone\code\datasets\processed")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    df_hc3_balanced = preprocess_hc3(CSV_PATH)
    
    if not df_hc3_balanced.empty:
        output_path = SAVE_DIR / "HC3_Clean_Balanced.csv"
        df_hc3_balanced.to_csv(output_path, index=False, encoding='utf-8')
        
        print("-" * 30)
        print(f"已保存均衡后的 HC3 数据: {output_path}")
        print(f"最终样本总数: {len(df_hc3_balanced)}")
    else:
        print("未提取到有效数据，请检查路径或文件格式。")

原始数据: 24322 行
清洗与过滤后样本数: 46971
>>> 类别均衡完成：Human=23158, AI=23158


C:\Users\21063\AppData\Local\Temp\ipykernel_32732\60118580.py:82: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  final_df = final_df.groupby('label').apply(


------------------------------
已保存均衡后的 HC3 数据: D:\Desktop\capstone\code\datasets\processed\HC3_Clean_Balanced.csv
最终样本总数: 46316


In [18]:
import pandas as pd
import re
from pathlib import Path

def clean_text_strict(text):
    """
    彻底清洗文本：只保留纯英文、数字和核心标点。
    """
    if not isinstance(text, str):
        return ""
    
    # 1. 去除 URL
    text = re.sub(r'http[s]?://\S+', '', text)
    
    # 2. 严格去除非英语符号 (白名单模式)
    # 仅保留 a-z, A-Z, 0-9, 空格及基础标点
    text = re.sub(r'[^a-zA-Z0-9\s.,!?;:\'"\-\(\)]', ' ', text)
    
    # 3. 合并多余空格和换行
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

def preprocess_llm_detect(csv_path):
    """
    预处理 LLM_Detect 数据集：严格清洗、长度截断、类别均衡 (1:1)
    """
    df = pd.read_csv(csv_path)
    print(f"原始数据: {len(df)} 行, 列: {df.columns.tolist()}")
    
    # 1. 严格清洗
    df['text'] = df['text'].apply(clean_text_strict)
    
    # 2. 构造标准字段 (竞赛中 generated 列即为 label)
    df['label'] = df['generated'].astype(int)
    df['source'] = 'LLMDetect_train'
    df['model'] = df['label'].apply(lambda x: 'gpt-3.5-turbo' if x == 1 else 'human')
    df['domain'] = 'essay'
    
    # 3. 长度过滤 (100 - 5000 字符)
    min_len, max_len = 100, 5000
    df = df[df['text'].str.len().between(min_len, max_len)].copy()
    
    # 4. 类别均衡 (Undersampling)
    label_counts = df['label'].value_counts()
    print(f"过滤后分布: {label_counts.to_dict()}")
    
    if len(label_counts) == 2:
        min_size = label_counts.min()
        # 强制 1:1 采样
        df_balanced = df.groupby('label').apply(
            lambda x: x.sample(n=min_size, random_state=42)
        ).reset_index(drop=True)
        # 随机打乱
        df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
        print(f">>> 均衡化完成：AI 和 Human 各保留 {min_size} 条样本")
    else:
        df_balanced = df
        print(">>> 警告：未检测到两个类别，无法均衡处理。")

    # 最终输出列
    final_df = df_balanced[['text', 'label', 'source', 'model', 'domain']].copy()
    return final_df

# ========================================
# 主程序
# ========================================
if __name__ == "__main__":
    CSV_PATH = r"D:\Desktop\capstone\code\datasets\LLM_Detect AI_train_essays.csv"
    SAVE_DIR = Path(r"D:\Desktop\capstone\code\datasets\processed")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    df_llm = preprocess_llm_detect(CSV_PATH)
    
    if not df_llm.empty:
        output_path = SAVE_DIR / "LLMDetect_Clean_Balanced.csv"
        df_llm.to_csv(output_path, index=False, encoding='utf-8')
        
        print("-" * 30)
        print(f"处理成功！保存路径: {output_path}")
        print(f"最终标签分布:\n{df_llm['label'].value_counts()}")
    else:
        print("处理后数据集为空，请检查原始文件内容。")

原始数据: 165767 行, 列: ['id', 'prompt_id', 'text', 'generated']
过滤后分布: {1: 127038, 0: 35026}
>>> 均衡化完成：AI 和 Human 各保留 35026 条样本


C:\Users\21063\AppData\Local\Temp\ipykernel_32732\1218244411.py:51: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_balanced = df.groupby('label').apply(


------------------------------
处理成功！保存路径: D:\Desktop\capstone\code\datasets\processed\LLMDetect_Clean_Balanced.csv
最终标签分布:
label
1    35026
0    35026
Name: count, dtype: int64


In [19]:
import pandas as pd
import re
from pathlib import Path

def clean_text_strict(text):
    """
    彻底清洗文本：只保留纯英文、数字和核心标点。
    """
    if not isinstance(text, str):
        return ""
    
    # 1. 去除 URL
    text = re.sub(r'http[s]?://\S+', '', text)
    
    # 2. 严格去除非英语符号 (白名单模式)
    # 仅保留 a-z, A-Z, 0-9, 空格及基础标点 (.,!?;:'"() -)
    text = re.sub(r'[^a-zA-Z0-9\s.,!?;:\'"\-\(\)]', ' ', text)
    
    # 3. 合并多余空格和换行
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

def parse_src(src):
    """解析 src 字段逻辑：cmv_human → domain='cmv', model='human'"""
    if pd.isna(src):
        return 'unknown', 'unknown'
    
    src = str(src).strip()
    parts = src.split('_')
    
    if len(parts) >= 2:
        domain = parts[0]
        model_part = '_'.join(parts[1:])
        if 'human' in model_part.lower():
            return domain, 'human'
        else:
            return domain, model_part
    else:
        return 'unknown', 'unknown'

def preprocess_mage(csv_path):
    """
    预处理 MAGE 数据集：严格清洗、长度过滤、1:1 类别均衡
    """
    df = pd.read_csv(csv_path)
    print(f"原始数据: {len(df)} 行, 列: {df.columns.tolist()}")
    
    # 1. 严格清洗文本
    df['text'] = df['text'].apply(clean_text_strict)
    
    # 2. 解析领域和模型
    df['domain'], df['model'] = zip(*df['src'].apply(parse_src))
    
    # 3. 构造 source 标识
    file_stem = Path(csv_path).stem
    split = file_stem.split('_')[-1] if '_' in file_stem else "data"
    df['source'] = df.apply(lambda row: f"MAGE_{split}_{row['domain']}", axis=1)
    
    # 4. 标签处理
    df['label'] = df['label'].astype(int)
    
    # 5. 长度过滤 (100 - 5000 字符)
    min_len, max_len = 100, 5000
    df = df[df['text'].str.len().between(min_len, max_len)].copy()
    
    # 6. 类别均衡 (1:1 比例采样)
    label_counts = df['label'].value_counts()
    print(f"清洗过滤后的分布: {label_counts.to_dict()}")
    
    if len(label_counts) == 2:
        min_size = label_counts.min()
        # 随机采样使 Human (0) 和 AI (1) 数量完全一致
        df_balanced = df.groupby('label').apply(
            lambda x: x.sample(n=min_size, random_state=42)
        ).reset_index(drop=True)
        # 随机打乱顺序
        df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
        print(f">>> 均衡化完成：AI 和 Human 各保留 {min_size} 条样本")
    else:
        df_balanced = df
        print(">>> 警告：类别不足，无法进行均衡处理。")

    # 最终输出标准字段
    final_df = df_balanced[['text', 'label', 'source', 'model', 'domain']].copy()
    return final_df

# ========================================
# 主程序
# ========================================
if __name__ == "__main__":
    # 请确保路径正确
    CSV_PATH = r"D:\Desktop\capstone\code\datasets\MAGE_test.csv"
    SAVE_DIR = Path(r"D:\Desktop\capstone\code\datasets\processed")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    df_mage = preprocess_mage(CSV_PATH)
    
    if not df_mage.empty:
        output_path = SAVE_DIR / "MAGE_test_Clean_Balanced.csv"
        df_mage.to_csv(output_path, index=False, encoding='utf-8')
        
        print("\n" + "="*30)
        print(f"处理成功！保存至: {output_path}")
        print(f"最终样本总数: {len(df_mage)}")
        print(f"最终标签分布:\n{df_mage['label'].value_counts()}")
        print("="*30)
    else:
        print("处理后数据集为空。")

原始数据: 60743 行, 列: ['text', 'label', 'src']
清洗过滤后的分布: {0: 29266, 1: 28253}
>>> 均衡化完成：AI 和 Human 各保留 28253 条样本


C:\Users\21063\AppData\Local\Temp\ipykernel_32732\3049846139.py:74: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_balanced = df.groupby('label').apply(



处理成功！保存至: D:\Desktop\capstone\code\datasets\processed\MAGE_test_Clean_Balanced.csv
最终样本总数: 56506
最终标签分布:
label
1    28253
0    28253
Name: count, dtype: int64


In [2]:
import pandas as pd
from pathlib import Path

# ==============================
# 1. 路径配置
# ==============================
PROCESSED_DIR = Path(r"D:\Desktop\capstone\code\datasets\processed")
SPLIT_SAVE_DIR = Path(r"D:\Desktop\capstone\code\datasets\splits")
SPLIT_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# ==============================
# 2. 数据集列表
# ==============================
files = [
    "MultiModel_Clean_Balanced.csv",
    "HC3_Clean_Balanced.csv",
    "LLMDetect_Clean_Balanced.csv",
    "MAGE_test_Clean_Balanced.csv",
    "DAIGT_Clean_Balanced.csv"
]

# ==============================
# 3. 参数
# ==============================
TRAIN_PER_CLASS = 2500
TEST_PER_CLASS  = 2500
RANDOM_STATE = 42

# ==============================
# 4. Split
# ==============================
for fname in files:
    print(f"\n=== Processing {fname} ===")
    df = pd.read_csv(PROCESSED_DIR / fname)

    # 自动识别 label
    label_col = "label" if "label" in df.columns else df.columns[-1]
    print(f"Using label column: {label_col}")

    # ==============================
    # Step 0: 文本级去重（关键！）
    # ==============================
    before = len(df)
    df = df.drop_duplicates(subset=["text"]).reset_index(drop=True)
    after = len(df)
    print(f"Deduplicated texts: {before} → {after}")

    # ==============================
    # Step 1: 测试集（先抽）
    # ==============================
    test_df = (
        df.groupby(label_col, group_keys=False)
          .apply(lambda x: x.sample(
              n=min(TEST_PER_CLASS, len(x)),
              random_state=RANDOM_STATE
          ))
          .reset_index(drop=True)
    )

    # ==============================
    # Step 2: 剩余数据中抽训练集
    # ==============================
    remaining_df = df[~df["text"].isin(test_df["text"])].reset_index(drop=True)

    train_df = (
        remaining_df.groupby(label_col, group_keys=False)
                    .apply(lambda x: x.sample(
                        n=min(TRAIN_PER_CLASS, len(x)),
                        random_state=RANDOM_STATE
                    ))
                    .reset_index(drop=True)
    )

    # ==============================
    # Step 3: 终极保险检查（文本级）
    # ==============================
    overlap = set(train_df["text"]).intersection(set(test_df["text"]))
    assert len(overlap) == 0, f"❌ Text leakage detected in {fname}"

    print("✔ Train/Test are strictly text-disjoint")

    # ==============================
    # Step 4: 保存
    # ==============================
    train_path = SPLIT_SAVE_DIR / fname.replace(".csv", "_TRAIN_5k.csv")
    test_path  = SPLIT_SAVE_DIR / fname.replace(".csv", "_TEST_5k.csv")

    train_df.to_csv(train_path, index=False, encoding="utf-8")
    test_df.to_csv(test_path, index=False, encoding="utf-8")

    print(f"Saved train: {len(train_df)} → {train_path.name}")
    print(f"Saved test : {len(test_df)} → {test_path.name}")

print("\n✅ All datasets split successfully with NO text leakage.")



=== Processing MultiModel_Clean_Balanced.csv ===
Using label column: label
Deduplicated texts: 142784 → 142409
✔ Train/Test are strictly text-disjoint
Saved train: 5000 → MultiModel_Clean_Balanced_TRAIN_5k.csv
Saved test : 5000 → MultiModel_Clean_Balanced_TEST_5k.csv

=== Processing HC3_Clean_Balanced.csv ===


C:\Users\21063\AppData\Local\Temp\ipykernel_44532\1491623567.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(
C:\Users\21063\AppData\Local\Temp\ipykernel_44532\1491623567.py:67: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(


Using label column: label
Deduplicated texts: 46316 → 44006
✔ Train/Test are strictly text-disjoint
Saved train: 5000 → HC3_Clean_Balanced_TRAIN_5k.csv
Saved test : 5000 → HC3_Clean_Balanced_TEST_5k.csv

=== Processing LLMDetect_Clean_Balanced.csv ===


C:\Users\21063\AppData\Local\Temp\ipykernel_44532\1491623567.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(
C:\Users\21063\AppData\Local\Temp\ipykernel_44532\1491623567.py:67: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(


Using label column: label
Deduplicated texts: 70052 → 69436
✔ Train/Test are strictly text-disjoint


C:\Users\21063\AppData\Local\Temp\ipykernel_44532\1491623567.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(
C:\Users\21063\AppData\Local\Temp\ipykernel_44532\1491623567.py:67: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(


Saved train: 5000 → LLMDetect_Clean_Balanced_TRAIN_5k.csv
Saved test : 5000 → LLMDetect_Clean_Balanced_TEST_5k.csv

=== Processing MAGE_test_Clean_Balanced.csv ===
Using label column: label
Deduplicated texts: 56506 → 55747
✔ Train/Test are strictly text-disjoint
Saved train: 5000 → MAGE_test_Clean_Balanced_TRAIN_5k.csv
Saved test : 5000 → MAGE_test_Clean_Balanced_TEST_5k.csv

=== Processing DAIGT_Clean_Balanced.csv ===


C:\Users\21063\AppData\Local\Temp\ipykernel_44532\1491623567.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(
C:\Users\21063\AppData\Local\Temp\ipykernel_44532\1491623567.py:67: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(


Using label column: label
Deduplicated texts: 34988 → 34987
✔ Train/Test are strictly text-disjoint
Saved train: 5000 → DAIGT_Clean_Balanced_TRAIN_5k.csv
Saved test : 5000 → DAIGT_Clean_Balanced_TEST_5k.csv

✅ All datasets split successfully with NO text leakage.


C:\Users\21063\AppData\Local\Temp\ipykernel_44532\1491623567.py:53: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(
C:\Users\21063\AppData\Local\Temp\ipykernel_44532\1491623567.py:67: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(
